### BOW- Bag Of Words

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "NLP is fun and amazing",
    "Machines understand NLP and Text",
    "Text processing is a part of NLP"
]

In [3]:
vectorizer = CountVectorizer()
x= vectorizer.fit_transform(corpus)

In [4]:
len(vectorizer.get_feature_names_out())

11

In [5]:
x.toarray()

array([[1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1],
       [0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0]])

In [6]:
# TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
x=vectorizer.fit_transform(corpus)

In [7]:
vectorizer.get_feature_names_out()

array(['amazing', 'and', 'fun', 'is', 'machines', 'nlp', 'of', 'part',
       'processing', 'text', 'understand'], dtype=object)

In [8]:
x.toarray()

array([[0.53409337, 0.40619178, 0.53409337, 0.40619178, 0.        ,
        0.31544415, 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 0.40619178, 0.        , 0.        , 0.53409337,
        0.31544415, 0.        , 0.        , 0.        , 0.40619178,
        0.53409337],
       [0.        , 0.        , 0.        , 0.35829137, 0.        ,
        0.27824521, 0.4711101 , 0.4711101 , 0.4711101 , 0.35829137,
        0.        ]])

In [9]:
import numpy as np
import math

In [10]:
tokenized_docs = [doc.lower().split() for doc in corpus]
tokenized_docs

[['nlp', 'is', 'fun', 'and', 'amazing'],
 ['machines', 'understand', 'nlp', 'and', 'text'],
 ['text', 'processing', 'is', 'a', 'part', 'of', 'nlp']]

In [13]:
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
vocab_index = {word: idx for idx, word in enumerate(vocab)}
print(vocab)
print(vocab_index)

['a', 'amazing', 'and', 'fun', 'is', 'machines', 'nlp', 'of', 'part', 'processing', 'text', 'understand']
{'a': 0, 'amazing': 1, 'and': 2, 'fun': 3, 'is': 4, 'machines': 5, 'nlp': 6, 'of': 7, 'part': 8, 'processing': 9, 'text': 10, 'understand': 11}


In [18]:
from collections import Counter
def compute_tf(doc_tokens):
  tf_vector = np.zeros(len(vocab))
  word_counts = Counter(doc_tokens)
  for word, count in word_counts.items():
    tf_vector[vocab_index[word]] = count / len(doc_tokens)
  return tf_vector

tf_matrix = np.array([compute_tf(doc) for doc in tokenized_docs])
tf_matrix

array([[0.        , 0.2       , 0.2       , 0.2       , 0.2       ,
        0.        , 0.2       , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.        , 0.        , 0.2       , 0.        , 0.        ,
        0.2       , 0.2       , 0.        , 0.        , 0.        ,
        0.2       , 0.2       ],
       [0.14285714, 0.        , 0.        , 0.        , 0.14285714,
        0.        , 0.14285714, 0.14285714, 0.14285714, 0.14285714,
        0.14285714, 0.        ]])

In [21]:
def compute_idf():
  N = len(tokenized_docs)
  idf = np.zeros(len(vocab))
  for word, idx in vocab_index.items():
    df=sum(1 for doc in tokenized_docs if word in doc)
    idf[idx] = math.log((N+1) / (df + 1)) + 1
  return idf

idf_vector = compute_idf()
print(idf_vector)


[1.69314718 1.69314718 1.28768207 1.69314718 1.28768207 1.69314718
 1.         1.69314718 1.69314718 1.69314718 1.28768207 1.69314718]


In [22]:
tfidf_matrix = tf_matrix * idf_vector
print(tfidf_matrix)

[[0.         0.33862944 0.25753641 0.33862944 0.25753641 0.
  0.2        0.         0.         0.         0.         0.        ]
 [0.         0.         0.25753641 0.         0.         0.33862944
  0.2        0.         0.         0.         0.25753641 0.33862944]
 [0.24187817 0.         0.         0.         0.18395458 0.
  0.14285714 0.24187817 0.24187817 0.24187817 0.18395458 0.        ]]


In [23]:
import pandas as pd
df = pd.DataFrame(tfidf_matrix, columns=vocab)
df

,a,amazing,and,fun,is,machines,nlp,of,part,processing,text,understand
0,0.000000,0.338629,0.257536,0.338629,0.257536,0.000000,0.200000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.257536,0.000000,0.000000,0.338629,0.200000,0.000000,0.000000,0.000000,0.257536,0.338629
2,0.241878,0.000000,0.000000,0.000000,0.183955,0.000000,0.142857,0.241878,0.241878,0.241878,0.183955,0.000000


## NLP Task-Feature Extraction

In [24]:
import numpy as np

feature_array = np.array(vectorizer.get_feature_names_out())
importance = np.argsort(x.toarray()).flatten()[::-1]

keywords = feature_array[importance[:5]]

In [25]:
print(keywords)

['processing' 'of' 'part' 'is' 'text']


In [32]:
corpus = [
    "NLP is fun and amazing",
    "Machines understand NLP and Text",
    "Text processing is a part of NLP"
]

In [38]:
top_n = 3
for i, row in enumerate(tfidf_matrix):
  top_indices = row.argsort()[-top_n:][::-1]
  keywords = [vocab[idx] for idx in top_indices]
  print(keywords)

['fun', 'amazing', 'and']
['understand', 'machines', 'and']
['part', 'processing', 'a']
